# Pandas vs Polars Karşılaştırması — Kanonik Sorgu

**Kanonik sorgu:** 2024 yılı iç hat uçuşları, `kalkis_havaalani` + `havayolu` bazında toplam gelir / ortalama gecikme / uçuş adedi (toplam gelire göre azalan). Aynı sorgu hem pandas (eager) hem Polars (lazy) ile uygulanır ve karşılaştırılır.

In [1]:
import pandas as pd
import polars as pl
import time

In [2]:
DATA_PATH = "data/ucus_verisi.csv"

df = pd.read_csv(DATA_PATH)

df.tail()

,ucus_no,kalkis_havaalani,varis_havaalani,havayolu,ucus_tipi,kalkis_tarihi,bilet_fiyati,gecikme_dk
149995,F149996,TZX,SZF,Pegasus,dis_hat,2024-04-12,2049.93,49
149996,F149997,ADB,ESB,Pegasus,ic_hat,2023-01-12,442.01,19
149997,F149998,SAW,ADB,THY,ic_hat,2024-01-01,4084.65,42
149998,F149999,SAW,SAW,AnadoluJet,dis_hat,2023-11-29,2095.93,21
149999,F150000,TZX,ESB,AnadoluJet,dis_hat,2023-12-12,3628.82,4


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ucus_no           150000 non-null  str    
 1   kalkis_havaalani  150000 non-null  str    
 2   varis_havaalani   150000 non-null  str    
 3   havayolu          150000 non-null  str    
 4   ucus_tipi         150000 non-null  str    
 5   kalkis_tarihi     150000 non-null  str    
 6   bilet_fiyati      150000 non-null  float64
 7   gecikme_dk        150000 non-null  int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 14.2 MB


In [4]:
df.shape

(150000, 8)

In [5]:
df["kalkis_tarihi"] = pd.to_datetime(df["kalkis_tarihi"])

df.head()

,ucus_no,kalkis_havaalani,varis_havaalani,havayolu,ucus_tipi,kalkis_tarihi,bilet_fiyati,gecikme_dk
0,F000001,IST,IST,AnadoluJet,ic_hat,2023-04-13,1712.75,22
1,F000002,VAN,ADB,SunExpress,ic_hat,2024-03-11,1548.61,1
2,F000003,IST,SAW,Pegasus,ic_hat,2023-09-28,3619.77,34
3,F000004,IST,GZT,THY,dis_hat,2023-04-17,1776.45,2
4,F000005,IST,AYT,THY,ic_hat,2023-03-13,4736.62,3


## Pandas Pipeline (kanonik sorgu)
Adil kıyas için pandas da CSV'yi **zamanlayıcı içinde** okur (Polars gibi I/O dahil).

In [6]:
def pandas_pipeline(data_path):
    # Pipeline süresini ölçmek için başlangıç zamanı.
    start = time.perf_counter()

    # Adil kıyas: CSV okuma da zamanlamaya dahil (Polars scan_csv ile simetrik).
    df_local = pd.read_csv(data_path)

    # Kanonik sorgu: 2024 yılı iç hat uçuşları.
    # CSV tarihi string okunur → str.startswith (Polars ile aynı semantik, na=False null'ları eler).
    result = (
        df_local[
            df_local["kalkis_tarihi"].str.startswith("2024", na=False) &
            (df_local["ucus_tipi"] == "ic_hat")
        ]
        # Havaalanı + havayolu bazında grupla.
        .groupby(["kalkis_havaalani", "havayolu"])
        # Toplam gelir, ortalama gecikme, uçuş adedi.
        .agg(
            toplam_gelir=("bilet_fiyati", "sum"),
            ort_gecikme_dk=("gecikme_dk", "mean"),
            ucus_adedi=("ucus_no", "count"),
        )
        .reset_index()
        # Gelire göre azalan sırala.
        .sort_values("toplam_gelir", ascending=False)
        .reset_index(drop=True)
    )

    end = time.perf_counter()
    elapsed_time = end - start

    # GİRDİ çerçevesinin bellek kullanımı (Polars ile aynı metrik için).
    memory_usage = df_local.memory_usage(deep=True).sum() / (1024**2)

    return result, elapsed_time, memory_usage

In [7]:
# Pandas pipeline'ı çalıştırıyorum (CSV yolunu veriyorum; okuma da zamanlamaya dahil).
pandas_result, pandas_time, pandas_memory = pandas_pipeline(DATA_PATH)

print("Pandas Süre:", round(pandas_time, 4), "saniye")
print("Pandas Bellek (girdi):", round(pandas_memory, 2), "MB")
pandas_result.head()

Pandas Süre: 0.0704 saniye
Pandas Bellek (girdi): 14.24 MB


,kalkis_havaalani,havayolu,toplam_gelir,ort_gecikme_dk,ucus_adedi
0,IST,THY,16962276.01,20.150703,6828
1,IST,Pegasus,12640409.14,19.314925,5052
2,SAW,THY,9771611.58,20.297015,3919
3,SAW,Pegasus,7389227.12,19.594282,2938
4,IST,AnadoluJet,6346649.80,20.150294,2555


## Polars Pipeline (lazy, kanonik sorgu)

In [8]:
def polars_pipeline(data_path):
    # Pipeline süresini ölçmek için başlangıç zamanı.
    start = time.perf_counter()

    # Kanonik sorgu — Polars LAZY (scan_csv ile veri hemen belleğe yüklenmez).
    # schema_overrides: küçük/boş dosyada sayısal tip çıkarımını sabitler.
    result = (
        pl.scan_csv(
            data_path,
            schema_overrides={"bilet_fiyati": pl.Float64, "gecikme_dk": pl.Int64},
        )
        # 2024 iç hat. CSV tarihi string okunur → str.starts_with.
        .filter(
            pl.col("kalkis_tarihi").str.starts_with("2024") &
            (pl.col("ucus_tipi") == "ic_hat")
        )
        .group_by(["kalkis_havaalani", "havayolu"])
        .agg([
            pl.col("bilet_fiyati").sum().alias("toplam_gelir"),
            pl.col("gecikme_dk").mean().alias("ort_gecikme_dk"),
            pl.len().alias("ucus_adedi"),
        ])
        .sort("toplam_gelir", descending=True)
        .collect()
    )

    end = time.perf_counter()
    elapsed_time = end - start

    # GİRDİ çerçevesinin bellek kullanımı (pandas ile aynı metrik için).
    memory_usage = pl.read_csv(data_path).estimated_size("mb")

    return result, elapsed_time, memory_usage

In [9]:
# Polars pipeline fonksiyonunu çalıştırıyorum.
# Fonksiyon bana:
# 1. İşlenmiş sonuç DataFrame'ini
# 2. Çalışma süresini
# 3. Bellek kullanımını döndürüyor.

polars_result, polars_time, polars_memory = polars_pipeline(DATA_PATH)

# Polars pipeline'ın toplam çalışma süresini saniye cinsinden yazdırıyorum.
print("Polars Süre:", round(polars_time, 4), "saniye")

# Polars sonuç DataFrame'inin yaklaşık bellek kullanımını MB cinsinden yazdırıyorum.
print("Polars Bellek:", round(polars_memory, 2), "MB")

# İşlenmiş sonuç DataFrame'inin ilk 5 satırını görüntülüyorum.
# Böylece aggregation işlemlerinin doğru çalışıp çalışmadığını kontrol edebiliyorum.
polars_result.head()

Polars Süre: 0.0075 saniye
Polars Bellek: 7.37 MB


kalkis_havaalani,havayolu,toplam_gelir,ort_gecikme_dk,ucus_adedi
str,str,f64,f64,u32
"""IST""","""THY""",1.6962e7,20.150703,6828
"""IST""","""Pegasus""",1.2640e7,19.314925,5052
"""SAW""","""THY""",9.7716e6,20.297015,3919
"""SAW""","""Pegasus""",7.3892e6,19.594282,2938
"""IST""","""AnadoluJet""",6346649.8,20.150294,2555


## Sonuç Eşitliği — Polars == pandas?
İki motorun aynı kanonik sorguda **birebir aynı sonucu** ürettiğini doğrularız.

In [10]:
# Polars ve pandas sonuçlarının AYNI olduğunu doğrula (kanonik sorgu).
import numpy as np

sutunlar = ["kalkis_havaalani", "havayolu", "toplam_gelir", "ort_gecikme_dk", "ucus_adedi"]
p = pandas_result[sutunlar].sort_values(["kalkis_havaalani", "havayolu"]).reset_index(drop=True)
q = (polars_result.to_pandas()[sutunlar]
     .sort_values(["kalkis_havaalani", "havayolu"]).reset_index(drop=True))
q["ucus_adedi"] = q["ucus_adedi"].astype("int64")   # Polars u32 → pandas int64

assert p.shape == q.shape, (p.shape, q.shape)
assert (p["ucus_adedi"] == q["ucus_adedi"]).all()
np.testing.assert_allclose(p["toplam_gelir"], q["toplam_gelir"], rtol=1e-9, atol=1e-6)
np.testing.assert_allclose(p["ort_gecikme_dk"], q["ort_gecikme_dk"], rtol=1e-9, atol=1e-6)
print("Polars ve pandas sonuçları EŞDEĞER ✅  (şekil:", p.shape, ")")

Polars ve pandas sonuçları EŞDEĞER ✅  (şekil: (50, 5) )


## Süre & Bellek Karşılaştırması
Her iki bellek değeri de **girdi çerçevesi** boyutudur (elma-elma karşılaştırma).

In [11]:
# Pandas ve Polars benchmark sonuçlarını karşılaştırmalı bir tabloya aktarıyorum.
comparison_df = pd.DataFrame({

    # Kullanılan veri işleme kütüphanelerini belirtiyorum.
    "Kutuphane": ["Pandas", "Polars"],

    # Her kütüphanenin pipeline çalışma süresini saniye cinsinden ekliyorum.
    "Sure (sn)": [pandas_time, polars_time],

    # Her kütüphanenin yaklaşık bellek kullanımını MB cinsinden ekliyorum.
    "Bellek (MB)": [pandas_memory, polars_memory]
})

# Oluşturulan benchmark karşılaştırma tablosunu görüntülüyorum.
comparison_df

,Kutuphane,Sure (sn),Bellek (MB)
0,Pandas,0.070376,14.240360
1,Polars,0.007522,7.373779


## Ölçekleme Testi (10K / 50K / 100K)
Her boyutta pandas ve Polars **aynı** örnek veriyle çalışır.

In [12]:
import tempfile, os

# Farklı veri boyutlarında benchmark.
sizes = [10_000, 50_000, 100_000]
benchmark_results = []

for size in sizes:
    # Ana veri setinden örnek al ve geçici CSV'ye yaz.
    sample_df = df.head(size)
    tmp = tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False).name
    sample_df.to_csv(tmp, index=False)

    # Her iki motor da AYNI geçici CSV'yi okur (tam simetrik, I/O dahil).
    _, pandas_time, pandas_memory = pandas_pipeline(tmp)
    _, polars_time, polars_memory = polars_pipeline(tmp)
    os.unlink(tmp)

    benchmark_results.append({
        "satir_sayisi": size,
        "pandas_sure_sn": round(pandas_time, 4),
        "polars_sure_sn": round(polars_time, 4),
        "pandas_bellek_mb": round(pandas_memory, 2),
        "polars_bellek_mb": round(polars_memory, 2),
    })

benchmark_df = pd.DataFrame(benchmark_results)
benchmark_df

,satir_sayisi,pandas_sure_sn,polars_sure_sn,pandas_bellek_mb,polars_bellek_mb
0,10000,0.0076,0.0033,0.95,0.49
1,50000,0.0251,0.0037,4.75,2.46
2,100000,0.0472,0.0048,9.49,4.92


In [13]:
# Pandas ve Polars çalışma sürelerini karşılaştırmak için
# hız farkı oranını hesaplıyorum.

benchmark_df["hiz_farki"] = (

    # Pandas çalışma süresini
    benchmark_df["pandas_sure_sn"]

    # Polars çalışma süresine bölüyorum.
    / benchmark_df["polars_sure_sn"]

# Sonucu daha okunabilir olması için 2 basamağa yuvarlıyorum.
).round(2)

# Güncellenmiş benchmark tablosunu görüntülüyorum.
benchmark_df

,satir_sayisi,pandas_sure_sn,polars_sure_sn,pandas_bellek_mb,polars_bellek_mb,hiz_farki
0,10000,0.0076,0.0033,0.95,0.49,2.30
1,50000,0.0251,0.0037,4.75,2.46,6.78
2,100000,0.0472,0.0048,9.49,4.92,9.83


## Ne Zaman Polars Mantıklı?

Polars; **büyük veri** (yüz binlerce–milyonlarca satır), **lazy evaluation** (sorguyu çalıştırmadan optimize etme) ve **çok çekirdekli paralellik** gerektiren işlerde pandas'a göre belirgin biçimde hızlıdır — bu notebook'ta aynı kanonik sorguda Polars, pandas'tan yaklaşık bir-iki kat (büyük veride çok daha fazla) hızlıdır. Örneğin günlük milyonlarca uçuş/log kaydını filtreleyip gruplayan **ETL** işlerinde Polars'ın lazy motoru gereksiz sütun ve satırları daha okumadan eler. Buna karşılık **küçük veri**, zengin **ekosistem entegrasyonu** (scikit-learn, statsmodels, matplotlib ile doğrudan uyum) ve **öğrenme kolaylığı** gereken durumlarda pandas hâlâ pratiktir; ekip pandas biliyorsa ve veri belleğe rahat sığıyorsa pandas yeterlidir. Özet: *büyük veri + tekrarlı/ETL → Polars; küçük veri + hızlı keşif + geniş kütüphane ihtiyacı → pandas.*